# Fase 4 + 5 (Gabungan) — Training & Evaluasi IndoBERT (Sentara)

Notebook **Run-All** satu sesi untuk Google Colab (GPU). Menggabungkan:
- **Fase 4**: full fine-tuning IndoBERT dengan konfigurasi final (LR 3e-5, batch 8, **2 epoch** — anti-overfitting per learning curve) + ekspor loss per epoch.
- **Fase 5**: evaluasi test set, cek overfitting, confusion matrix, learning curve, gate.

> **Sengaja DILEWATI untuk hemat GPU** (sudah selesai & ter-commit di run Fase 4 sebelumnya): focused random search (FR-4.6) dan 5-fold cross-validation (FR-4.7). Ringkasan CV tetap dipakai di laporan Fase 5 via `cross_validation_report.json`.

**Cara pakai:** set Runtime ▸ T4 GPU, lalu **Runtime ▸ Run all**. Estimasi ~25–45 menit (mayoritas di training).

Metrik utama: **macro F1** (target ≥ 0.85). Class weight wajib (imbalance kelas > 15%).

## 1. Setup environment
Clone repo + install dependensi (pinning `transformers<5` agar API Trainer konsisten).

In [ ]:
# Clone repo proyek
!git clone https://github.com/rahmatullahaditya780/sentiment-analysis-indobert.git
%cd sentiment-analysis-indobert

!pip -q install "transformers>=4.40,<5" "torch>=2.2.0" "datasets>=2.19.0" "accelerate>=0.30.0" "scikit-learn>=1.4.0" "pandas>=2.2.0" "matplotlib>=3.9.0"

# Alternatif (mount Drive, bila tak via GitHub):
# from google.colab import drive; drive.mount("/content/drive")
# %cd /content/drive/MyDrive/SKRIPSI/sentiment-analysis-indobert

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('CUDA tersedia:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# === Toggle: pakai data AUGMENTED (back-translation kelas netral) ===
# True  -> re-train pakai clean_train_augmented.csv; output ke path *_augmented
#          sehingga best_model & evaluation_final.json BASELINE tidak tertimpa
#          (A/B comparison yang jujur untuk laporan/sidang).
# False -> perilaku baseline (run Fase 4/5 sebelumnya, neutral F1 0.817).
USE_AUGMENTED_TRAIN = True

from src.modeling.config import MODELS_DIR, REPORTS_DIR

if USE_AUGMENTED_TRAIN:
    RUN_TAG = 'augmented'
    MODEL_OUT_DIR = MODELS_DIR / 'best_model_augmented'
    EVAL_REPORT_PATH = REPORTS_DIR / 'evaluation_final_augmented.json'
else:
    RUN_TAG = 'baseline'
    MODEL_OUT_DIR = MODELS_DIR / 'best_model'
    EVAL_REPORT_PATH = REPORTS_DIR / 'evaluation_final.json'

print('RUN_TAG       :', RUN_TAG)
print('Model output  :', MODEL_OUT_DIR)
print('Eval report   :', EVAL_REPORT_PATH)

## 2. Load data bersih (output Fase 3)
`clean_train/validation/test.csv` (kolom: text, label, source). Pada mode augmented, train memakai `clean_train_augmented.csv`.

In [ ]:
from src.modeling.data import load_clean_split, label_distribution, compute_class_weights
from src.modeling.config import LABEL2ID, DataConfig

# Train mengikuti toggle (augmented bila aktif); validation & test SELALU asli
# agar evaluasi mencerminkan distribusi nyata (jujur secara metodologis).
data_cfg = DataConfig(use_augmented_train=USE_AUGMENTED_TRAIN)
train_df = load_clean_split('train', data_cfg)
val_df = load_clean_split('validation', data_cfg)
test_df = load_clean_split('test', data_cfg)

print('train/val/test:', len(train_df), len(val_df), len(test_df))
print('LABEL2ID:', LABEL2ID)
print('Distribusi train:', label_distribution(train_df))
print('Class weight:', [round(w, 3) for w in compute_class_weights(train_df)])

## 3. Full fine-tuning (konfigurasi final Fase 4)
Latih pada **full training set** dengan config final (LR 3e-5, batch 8, **2 epoch**), early stopping (patience=2) + LR scheduler linear+warmup. Output ke `MODEL_OUT_DIR` (mode augmented → `models/best_model_augmented/`, tidak menimpa baseline). Loss per epoch diekspor ke `outputs/logs/training_log.csv` untuk learning curve Fase 5.

In [ ]:
from src.modeling.config import TrainingConfig, ensure_output_dirs
from src.modeling.data import build_hf_dataset, compute_class_weights
from src.modeling.trainer import train_model, export_loss_history_csv
from src.preprocessing.tokenizer_wrapper import IndoBERTTokenizerWrapper

ensure_output_dirs()
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Konfigurasi final: 2 epoch (titik optimal val loss dari learning curve 3-epoch;
# mengurangi overfitting -> gap F1 train vs val). LR 3e-5, batch 8.
final_cfg = TrainingConfig(learning_rate=3e-5, batch_size=8, num_epochs=2, run_name=f'final-{RUN_TAG}')

tokenizer = IndoBERTTokenizerWrapper().tokenizer
# Class weight dihitung dari train aktif: pada run augmented, bobot netral
# otomatis mengecil (sampelnya bertambah) -> augmentasi + weight saling melengkapi.
class_weights = compute_class_weights(train_df)
train_ds = build_hf_dataset(train_df, tokenizer=tokenizer)
val_ds = build_hf_dataset(val_df, tokenizer=tokenizer)

trainer, val_metrics = train_model(
    final_cfg,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    class_weights=class_weights,
    output_dir=MODEL_OUT_DIR,
    tokenizer=tokenizer,
    save_model=True,
)
print('Validation metrics:', val_metrics)

# Ekspor loss per epoch untuk learning curve Fase 5 (FR-5.6).
log_path = export_loss_history_csv(trainer)
print('Training log ->', log_path)

## 4. Evaluasi test set + cek overfitting (Fase 5, FR-5.1 s/d FR-5.5)
Muat model dari run aktif (`MODEL_OUT_DIR`), prediksi test set (metrik final + confusion matrix), serta train & validation untuk cek overfitting (gap F1 train vs val ≤ 5%). Report ditulis ke `EVAL_REPORT_PATH` (mode augmented → `evaluation_final_augmented.json`, tidak menimpa baseline).

In [ ]:
from src.evaluation.evaluator import load_best_model, predict_split
from src.evaluation.metrics import build_evaluation_report, compute_classification_metrics, overfitting_gap
from src.evaluation.cross_val_report import summarize_cv
import json

# Muat model dari run aktif (baseline atau augmented).
model, tok = load_best_model(MODEL_OUT_DIR)

# Prediksi test set -> metrik final + confusion matrix.
y_true, y_pred, y_proba = predict_split(test_df, model, tok)

# Prediksi train & validation -> cek overfitting.
tr_true, tr_pred, _ = predict_split(train_df, model, tok)
va_true, va_pred, _ = predict_split(val_df, model, tok)
train_f1 = compute_classification_metrics(tr_true, tr_pred)['f1_macro']
val_f1 = compute_classification_metrics(va_true, va_pred)['f1_macro']

cv_summary = summarize_cv()
overfit = overfitting_gap(train_f1=train_f1, val_f1=val_f1)

# write=False lalu tulis manual ke EVAL_REPORT_PATH agar laporan BASELINE
# (evaluation_final.json) tidak tertimpa pada run augmented.
report = build_evaluation_report(y_true, y_pred, cv_summary=cv_summary, overfitting=overfit, write=False)
EVAL_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(EVAL_REPORT_PATH, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print('Report ->', EVAL_REPORT_PATH)
print(json.dumps(report, indent=2, ensure_ascii=False))

## 5. Confusion matrix (FR-5.5)

In [ ]:
from src.evaluation.visualizer import plot_confusion_matrix

cm_path = plot_confusion_matrix(report['confusion_matrix']['matrix'], normalize=False)
print('Tersimpan:', cm_path)

from IPython.display import Image
Image(str(cm_path))

## 6. Learning curve (FR-5.6)
Training loss vs validation loss per epoch dari `training_log.csv` (Bagian 3).

In [ ]:
from src.evaluation.visualizer import plot_learning_curve

lc_path = plot_learning_curve()
print('Tersimpan:', lc_path)

from IPython.display import Image
Image(str(lc_path))

## 7. Cek gate Fase 5
Validasi: accuracy & macro F1 ≥ 0.85, semua artefak tersedia.

In [ ]:
import subprocess, sys
res = subprocess.run([sys.executable, '-m', 'scripts.validate_phase5_gate'], capture_output=True, text=True)
print(res.stdout)
print(res.stderr)

## 8. Unduh artefak
Arsipkan deliverable Fase 5 (laporan + chart + log) lalu unduh. Taruh isi `outputs/` ke proyek lokal di path yang sama.

## 9. Perbandingan Baseline vs Augmented (per-kelas)
Bandingkan F1 per-kelas (fokus **neutral**) antara baseline (`evaluation_final.json`) dan run augmented (`evaluation_final_augmented.json`). Tabel ini langsung dipakai untuk laporan/sidang sebagai bukti efek back-translation pada kelas minoritas.

In [ ]:
import json
from pathlib import Path
from src.modeling.config import REPORTS_DIR

base_p = REPORTS_DIR / 'evaluation_final.json'              # baseline (ter-commit)
aug_p = REPORTS_DIR / 'evaluation_final_augmented.json'     # hasil run augmented

def per_class_f1(path):
    if not Path(path).exists():
        return None
    r = json.load(open(path, encoding='utf-8'))
    pc = r.get('per_class', {})
    out = {lbl: pc.get(lbl, {}).get('f1') for lbl in ('negative', 'neutral', 'positive')}
    out['f1_macro'] = r.get('metrics', {}).get('f1_macro')
    return out

base, aug = per_class_f1(base_p), per_class_f1(aug_p)
print(f"{'kelas':<12}{'baseline':>12}{'augmented':>12}{'delta':>10}")
for lbl in ('negative', 'neutral', 'positive', 'f1_macro'):
    b = (base or {}).get(lbl); a = (aug or {}).get(lbl)
    d = (a - b) if (a is not None and b is not None) else None
    bs = f'{b:.4f}' if b is not None else '-'
    as_ = f'{a:.4f}' if a is not None else '-'
    ds = f'{d:+.4f}' if d is not None else '-'
    print(f'{lbl:<12}{bs:>12}{as_:>12}{ds:>10}')
print('\nFokus: kenaikan F1 kelas NEUTRAL = bukti efek back-translation.')

In [ ]:
# Artefak Fase 5 (kecil) — laporan baseline + augmented + chart + log.
!zip -r artefak_fase5.zip outputs/reports/evaluation_final.json outputs/reports/evaluation_final_augmented.json outputs/charts outputs/logs/training_log.csv
from google.colab import files
files.download('artefak_fase5.zip')

# Opsional: arsipkan model augmented (442 MB) bila ingin menyimpan ulang.
# !zip -r best_model_augmented.zip models/best_model_augmented
# files.download('best_model_augmented.zip')